# Base and tuned replies on a Kaggle T4 (generation only)

Generates every reply the one-command eval would score: the base model with the zero-shot prompt
and with the best ablation prompt, and the tuned adapter `hf:HUB_REPO`, over `EVAL_SET`. It runs
`eval.py --generate-only`, so it makes no judge calls and needs no API key. Judge the replies
locally:

```sh
uv run eval.py --judge-from <run_dir>/generations.jsonl
```

That writes the same `results.json`, `results.md` and `transcripts.jsonl` as the one-command
eval. `generation_pins.json` records the eval set hash, model revisions, chat-template hash,
decoding and library versions.

Before running:

- Settings: accelerator **GPU T4 x2** (the notebook uses one of the two), **Internet on**.
- Add-ons, Secrets: `HF_TOKEN` (a read token; the adapter repo is private) and `REPO_URL` (the
  code repo's clone URL). Neither secret is printed, and the notebook stops before doing anything
  when one is missing or empty.
- `HUB_REPO` is set. Check the line marked `# OWNER:` in the parameters cell (the eval set).

Afterwards, download the run directory under `/kaggle/working/eval` (both files) and run
`--judge-from` on it in the repo checkout at `REPO_COMMIT`.

In [ ]:
# The code repo is cloned at REPO_COMMIT; its clone URL is the REPO_URL Kaggle secret.
REPO_COMMIT = "3ff1c63efd9cb485a1de709bd2ffc4f56d4f3c42"
# The tuned adapter on the (private) Hub repo; append @<revision> to pin a commit.
HUB_REPO = "joe-alfaro/groundcite-qwen3-1.7b"
# OWNER: the eval set, relative to the repo at REPO_COMMIT.
EVAL_SET = "data/scenarios/heldout.jsonl"
OUT = "/kaggle/working/eval"

The required `HF_TOKEN` (for the private adapter repo) and `REPO_URL` secrets; nothing is printed.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()


def required(label: str, read) -> str:
    """A Kaggle secret's value; stops with a clear message when it is not attached or empty."""
    try:
        value = read()
    except Exception:  # Kaggle raises a raw service error for a secret that is not attached
        value = ""
    if not value:
        raise RuntimeError(
            f"the Kaggle secret {label} is missing or empty: add it under Add-ons > Secrets "
            "and attach it to this notebook"
        ) from None
    return value


# The adapter repo is private, so the token is required.
os.environ["HF_TOKEN"] = required("HF_TOKEN", lambda: client.get_secret("HF_TOKEN"))
# The code repo's clone URL; it is kept out of every tracked file.
REPO_URL = required("REPO_URL", lambda: client.get_secret("REPO_URL"))

Check the GPU, then clone the repo at the pinned commit and install around Kaggle's torch.

In [ ]:
import os

import torch

# Kaggle's "GPU T4 x2" shows two GPUs; the model runs on one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if not torch.cuda.is_available():
    raise RuntimeError("no CUDA GPU: set Settings > Accelerator to GPU T4 x2")
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
import re
import subprocess
import tempfile
from importlib import metadata
from urllib.parse import urlsplit

REPO_DIR = "/tmp/repo"


def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)


def clone(url: str, dest: str) -> None:
    """Clone without ever showing the URL: a private repo's URL can carry a token. git's error is
    kept for debugging, but git may quote the URL, its host or its path, so all are redacted."""
    result = subprocess.run(["git", "clone", "--quiet", url, dest], capture_output=True, text=True)
    if result.returncode != 0:
        parts = urlsplit(url)
        secret = {url, parts.netloc, parts.path, parts.path.strip("/"), *re.split(r"[@:/]+", url)}
        secret |= {parts.hostname or "", parts.username or "", parts.password or ""}
        detail = result.stderr or ""
        for piece in sorted((p for p in secret if len(p) > 2), key=len, reverse=True):
            detail = detail.replace(piece, "<redacted>")
        raise RuntimeError(
            f"git clone of the REPO_URL secret failed (exit {result.returncode}): {detail.strip()}"
        ) from None


clone(REPO_URL, REPO_DIR)
sh(f"git -C {REPO_DIR} checkout --quiet {REPO_COMMIT}")
# An explicit pin list without torch, triton or nvidia-*: Kaggle's CUDA torch stays. A transitive
# requirement (xformers via Unsloth) could still pull in another torch, so the preinstalled torch
# stack is pinned as a constraint: pip then picks compatible versions or fails loudly.
pins = []
for name in ("torch", "torchvision", "torchaudio", "triton"):
    try:
        pins.append(f"{name}=={metadata.version(name)}\n")
    except metadata.PackageNotFoundError:
        pass
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as constraints:
    constraints.writelines(pins)
requirements = f"{REPO_DIR}/train/requirements-kaggle.txt"
sh(f"pip install --quiet -r {requirements} -c {constraints.name}")
sh(f"pip install --quiet --no-deps -e {REPO_DIR}")

Generate the base and tuned replies (fp16 on the T4, greedy, thinking off).

In [ ]:
generate = f"python eval.py --model hf:{HUB_REPO} --eval-set {EVAL_SET} --generate-only"
sh(f"cd {REPO_DIR} && {generate} --out {OUT}")